# 🧬 Conformal Prediction for Trustworthy Biomedical Models
## Distribution-free prediction intervals, classification sets, subgroup reliability, and distribution shift

### Overview

A biomedical model may achieve a good average RMSE or classification accuracy and still be difficult to trust for a new patient. A point prediction does not say how uncertain that prediction is, and a probability score alone does not guarantee that uncertainty statements are correct for future patients.

This tutorial extends familiar ideas—held-out validation data, residuals, regression, classification, population structure, and cross-cohort evaluation—into a conformal prediction workflow:

~~~text
Point model → Calibration scores → Prediction interval or set → Coverage audit → Shift stress test
~~~

Every patient record is generated inside this notebook. No external data file, image, pretrained model, internet connection, or GPU is required. The simulated cohort is a teaching model, not evidence about a real patient population or cancer mechanism.

### Learning Objectives

By the end of this tutorial, you will be able to:

1. distinguish training, calibration, and test data;
2. explain exchangeability and marginal coverage in plain language;
3. implement split conformal regression from calibration residuals;
4. compare coverage with interval efficiency;
5. construct locally adaptive intervals and audit subgroup coverage;
6. build multiclass conformal prediction sets;
7. compare marginal and class-conditional calibration;
8. diagnose why coverage may fail under distribution shift.

### Tutorial Roadmap

| Part | Topic | Main output |
| --- | --- | --- |
| 0 | Biomedical uncertainty and a controlled cohort | Self-contained patient data |
| 1 | Split conformal regression | Distribution-free prediction intervals |
| 2 | Adaptive intervals and subgroup reliability | Group-wise coverage and width audit |
| 3 | Classification prediction sets | Marginal and class-conditional sets |
| 4 | Distribution-shift stress test | Coverage under a new cohort |
| 5 | Biomedical interpretation | Deployment checklist and limitations |

### Core References

The notation, split conformal workflow, and distinction between **coverage** and **efficiency** mainly follow:

1. Angelopoulos, A. N., & Bates, S. (2022). **A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification.**  
   https://arxiv.org/abs/2107.07511

The foundational explanations of **exchangeability** and **conformal validity** are supplemented by:

2. Shafer, G., & Vovk, V. (2008). **A Tutorial on Conformal Prediction.** *Journal of Machine Learning Research*, 9, 371–421.  
   https://jmlr.org/papers/v9/shafer08a.html

### Runtime and Prerequisites

- **Hardware:** CPU only
- **Expected runtime:** under one minute on a typical laptop
- **Libraries:** NumPy, pandas, Matplotlib
- **Prerequisites:** basic Python, regression residuals, classification probabilities, and train/test splits

> Conformal prediction is distribution-free, not assumption-free. The classical guarantee in this notebook relies on exchangeability between calibration and future test examples.


In [ ]:
# Part 0.1 — Environment setup
# ============================================================
# Only standard scientific Python libraries are required.
# A fixed random seed makes every simulated cohort reproducible.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

COLORS = {
    "blue": "#4C78A8",
    "orange": "#F58518",
    "red": "#E45756",
    "green": "#54A24B",
    "purple": "#B279A2",
    "gray": "#9D9D9D",
}

print("Environment setup completed.")
print("NumPy version:", np.__version__)
print("No external data, internet connection, or GPU is required.")


# Part 0 — From Model Evaluation to Predictive Uncertainty

## One Cohort, Two Prediction Problems

A translational genomics group is studying a patient cohort collected across several clinical sites. Each patient has:

- age and population-group metadata;
- five latent pathway activities;
- 40 correlated gene-expression measurements;
- a continuous inflammation-response phenotype;
- one of five simulated cancer subtypes.

The subtype names end in **“-like”** because these are synthetic teaching labels, not clinical diagnoses.

We will use the same cohort for two questions:

1. **Regression:** What range of phenotype values is plausible for a new patient?
2. **Classification:** Which cancer subtypes remain plausible for a new expression profile?

The generator deliberately includes pathway correlation, subgroup imbalance, heteroscedastic noise, and a controllable site effect. Because the mechanisms are known, we can determine why a coverage failure occurs.

## Three Data Roles

| Split | Role | May it fit the point model? | May it choose the conformal threshold? | May it report final performance? |
| --- | --- | ---: | ---: | ---: |
| Training | Learn the prediction rule | Yes | No | No |
| Calibration | Convert errors into uncertainty | No | Yes | No |
| Test | Evaluate unseen predictions | No | No | Yes |

A calibration set is not an ordinary validation set used repeatedly for model selection. Once it is used to choose a conformal threshold, it must remain separate from final testing.

## Coverage and Efficiency

For a prediction set $C(X)$ and miscoverage level $\alpha$,

$$
\Pr\{Y_{\mathrm{new}}\in C(X_{\mathrm{new}})\}\geq 1-\alpha.
$$

This is a **marginal** repeated-sampling statement. It does not mean that every subgroup or every individual feature value receives exactly the same coverage.

- **Validity:** Does the procedure attain its target coverage?
- **Efficiency:** How narrow are regression intervals, or how small are classification sets?

A method that always returns the entire outcome space may cover perfectly but provide little useful information.


In [ ]:
# Part 0.2 — Define one self-contained biomedical cohort generator
# ============================================================
# The generator creates pathway activities first, then correlated genes,
# a continuous phenotype, and a five-class subtype label.

SUBTYPE_NAMES = np.array([
    "BRCA-like", "KIRC-like", "COAD-like", "LUAD-like", "PRAD-like"
])
GROUP_NAMES = np.array(["Group A", "Group B", "Group C"])
PATHWAY_NAMES = np.array([
    "Proliferation", "Immune", "Stromal", "Metabolism", "Hypoxia"
])

# Rows are subtype-specific pathway means. The profiles overlap on purpose:
# a realistic uncertainty exercise needs some ambiguous patients.
SUBTYPE_PROFILES = np.array([
    [ 1.10,  0.10,  0.35, -0.10,  0.20],
    [ 0.20,  1.10, -0.10,  0.45,  0.75],
    [ 0.65,  0.35,  1.05,  0.10, -0.20],
    [ 0.85,  0.20,  0.10,  0.70,  1.00],
    [ 0.25, -0.10,  0.55,  1.05,  0.10],
])

GROUP_PATHWAY_EFFECTS = np.array([
    [ 0.00,  0.00,  0.00,  0.00,  0.00],
    [ 0.10, -0.15,  0.05,  0.10,  0.00],
    [-0.10,  0.20,  0.15, -0.05,  0.20],
])

# Fixed gene loadings are shared by source and shifted cohorts.
loading_rng = np.random.default_rng(2025)
N_GENES = 40
GENE_LOADINGS = loading_rng.normal(0, 0.55, size=(N_GENES, len(PATHWAY_NAMES)))
for pathway_id in range(len(PATHWAY_NAMES)):
    # Give each pathway a small block of strongly related genes.
    start = pathway_id * 8
    GENE_LOADINGS[start:start + 8, pathway_id] += 1.10

SITE_PATHWAY_VECTOR = np.array([0.20, -0.10, 0.15, 0.10, 0.25])
SHIFT_GENE_VECTOR = loading_rng.normal(0, 1.0, size=N_GENES)


def generate_biomedical_cohort(n_patients, seed, shift_strength=0.0):
    '''Generate a source cohort or a controllably shifted future cohort.'''
    local_rng = np.random.default_rng(seed)

    # Shift changes subtype prevalence, group composition, and site prevalence.
    base_subtype_probs = np.array([0.27, 0.22, 0.19, 0.18, 0.14])
    shifted_subtype_probs = np.array([0.15, 0.18, 0.18, 0.30, 0.19])
    subtype_probs = (
        (1 - shift_strength) * base_subtype_probs
        + shift_strength * shifted_subtype_probs
    )
    subtype_probs = np.clip(subtype_probs, 0.01, None)
    subtype_probs = subtype_probs / subtype_probs.sum()

    base_group_probs = np.array([0.55, 0.30, 0.15])
    shifted_group_probs = np.array([0.30, 0.35, 0.35])
    group_probs = (
        (1 - shift_strength) * base_group_probs
        + shift_strength * shifted_group_probs
    )
    group_probs = np.clip(group_probs, 0.01, None)
    group_probs = group_probs / group_probs.sum()

    subtype = local_rng.choice(len(SUBTYPE_NAMES), size=n_patients, p=subtype_probs)
    group = local_rng.choice(len(GROUP_NAMES), size=n_patients, p=group_probs)
    site_probability = np.clip(0.35 + 0.35 * shift_strength, 0.0, 0.95)
    site = local_rng.binomial(1, site_probability, size=n_patients)

    age = np.clip(
        local_rng.normal(56 + 2.0 * subtype - 1.5 * group, 10.5, n_patients),
        25,
        85,
    )

    pathway_noise = local_rng.normal(
        0, 0.82 + 0.10 * shift_strength,
        size=(n_patients, len(PATHWAY_NAMES)),
    )
    pathways = (
        SUBTYPE_PROFILES[subtype]
        + GROUP_PATHWAY_EFFECTS[group]
        + site[:, None] * SITE_PATHWAY_VECTOR
        + pathway_noise
    )

    # The new site produces a feature-dependent measurement batch effect.
    genes = pathways @ GENE_LOADINGS.T
    genes += local_rng.normal(0, 0.75, size=(n_patients, N_GENES))
    genes += shift_strength * site[:, None] * 0.55 * SHIFT_GENE_VECTOR

    # The continuous outcome has a nonlinear mean and heteroscedastic noise.
    phenotype_mean = (
        50
        + 5.0 * pathways[:, 0]
        - 3.8 * pathways[:, 1]
        + 2.5 * pathways[:, 2]
        + 2.2 * pathways[:, 3] * pathways[:, 4]
        + 0.12 * (age - 55)
        + 0.9 * (group == 1)
        - 0.6 * (group == 2)
    )
    noise_sd = (
        2.2
        + 0.9 * np.abs(pathways[:, 4])
        + 1.0 * (group == 1)
        + 2.4 * (group == 2)
        + 1.4 * shift_strength
    )
    phenotype = phenotype_mean + local_rng.normal(0, noise_sd)

    frame = pd.DataFrame({
        "patient_id": [f"P{i:04d}" for i in range(n_patients)],
        "age": age,
        "population_group": GROUP_NAMES[group],
        "group_code": group,
        "site": site,
        "subtype": subtype,
        "subtype_name": SUBTYPE_NAMES[subtype],
        "phenotype": phenotype,
        "true_noise_sd": noise_sd,
    })
    for gene_id in range(N_GENES):
        frame[f"gene_{gene_id:02d}"] = genes[:, gene_id]

    return frame


cohort = generate_biomedical_cohort(
    n_patients=2400,
    seed=SEED,
    shift_strength=0.0,
)

print("Cohort shape:", cohort.shape)
print("Gene-expression features:", N_GENES)
print("Phenotype range:", tuple(np.round(cohort["phenotype"].agg(["min", "max"]), 2)))
display(cohort.head())


In [ ]:
# Part 0.3 — Inspect cohort composition and controlled uncertainty
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cohort["subtype_name"].value_counts().reindex(SUBTYPE_NAMES).plot.bar(
    ax=axes[0], color=COLORS["blue"]
)
axes[0].set_title("Simulated cancer subtypes")
axes[0].set_xlabel("")
axes[0].set_ylabel("Patients")
axes[0].tick_params(axis="x", rotation=40)

cohort["population_group"].value_counts().reindex(GROUP_NAMES).plot.bar(
    ax=axes[1], color=COLORS["orange"]
)
axes[1].set_title("Population-group imbalance")
axes[1].set_xlabel("")
axes[1].set_ylabel("Patients")
axes[1].tick_params(axis="x", rotation=0)

for group_name, color in zip(GROUP_NAMES, [COLORS["blue"], COLORS["green"], COLORS["red"]]):
    values = cohort.loc[
        cohort["population_group"] == group_name, "phenotype"
    ]
    axes[2].hist(values, bins=28, alpha=0.48, label=group_name, color=color)
axes[2].set_title("Phenotype distributions overlap")
axes[2].set_xlabel("Continuous phenotype")
axes[2].set_ylabel("Patients")
axes[2].legend()

plt.tight_layout()
plt.show()

noise_summary = cohort.groupby("population_group", observed=True)["true_noise_sd"].agg(
    ["count", "mean", "median"]
)
display(noise_summary.round(3))

assert not cohort.isna().any().any()
assert set(cohort["subtype_name"].unique()) == set(SUBTYPE_NAMES)
assert cohort["patient_id"].is_unique
print("Checkpoint passed: the self-contained cohort is complete and reproducible.")


# Part 1 — Split Conformal Regression

## Concept — From Residuals to Distribution-Free Prediction Intervals

### Point prediction is not predictive uncertainty

A regression model estimates a location such as the conditional mean,

$$
\widehat\mu(x)\approx\mathbb E[Y\mid X=x].
$$

RMSE or mean absolute error summarizes how point predictions perform across a dataset, but neither quantity gives an uncertainty interval for one future response. A confidence interval for a regression coefficient or for the conditional mean answers a different question: it describes uncertainty about an estimated parameter or mean, not the range of a new patient outcome.

Split conformal regression turns held-out prediction errors into a **prediction interval** without assuming that the response or residuals follow a Gaussian distribution.

### Nonconformity scores

Conformal prediction begins by deciding what makes an observed outcome unusual relative to a fitted model. For ordinary regression, a natural nonconformity score is the absolute residual

$$
R_i=|Y_i-\widehat\mu(X_i)|.
$$

A small score means that the observed phenotype agrees closely with the point prediction. A large score means that the observation is difficult for the fitted model to predict. Conformal prediction does not require the point model to be correct; a better point model generally produces smaller residuals and therefore more efficient intervals.

### The split-conformal workflow

The procedure separates model fitting from uncertainty calibration:

1. fit $\widehat\mu$ using the **training set**;
2. predict the outcomes in the **calibration set**;
3. calculate one nonconformity score for each calibration patient;
4. select a high empirical score quantile $q$;
5. return $[\widehat\mu(x)-q,\widehat\mu(x)+q]$ for a new patient.

The calibration outcomes never update the point-model coefficients. Their role is to determine how large a future residual may be while respecting the requested error rate.

### Why the conformal quantile is corrected

For $n_{\mathrm{cal}}$ calibration scores and miscoverage level $\alpha$, split conformal uses the order-statistic rank

$$
k=\left\lceil(n_{\mathrm{cal}}+1)(1-\alpha)\right\rceil.
$$

The extra $+1$ accounts for the unseen test score that conceptually joins the calibration scores. When this rank is supported by the calibration sample, $q$ is the $k$-th smallest calibration score. Very small $\alpha$ may require more calibration examples; otherwise only an uninformative infinite interval can provide the requested finite-sample guarantee.

### What is guaranteed—and what is not

If calibration and future test examples are exchangeable, the rank of the future score among calibration and test scores is symmetric. This produces finite-sample **marginal coverage**:

$$
\Pr\{Y_{\mathrm{new}}\in C(X_{\mathrm{new}})\}\geq 1-\alpha.
$$

The result is model-agnostic: linear regression, a tree ensemble, or a neural network can all be wrapped by the same calibration step. However, the guarantee averages over future patients. It does not promise exactly $1-\alpha$ coverage for every population group, every feature value, or every individual.

### Coverage and efficiency must be read together

- **Coverage** asks how often the interval contains the observed outcome.
- **Efficiency** asks how informative the interval is, usually through its width.

An interval spanning every possible phenotype would cover nearly everything but provide little decision value. Conformal evaluation therefore reports coverage and width together.

---

## Task 1 — Reserve a Calibration Cohort

### Task background

The research group wants to predict a continuous inflammation-response phenotype from gene-expression and demographic features. The point model needs training patients, while conformal calibration needs a separate group of patients whose residuals behave like residuals from future patients.

### What you will do

Divide the source cohort into mutually exclusive training, calibration, and test sets. Preserve all five subtype classes in every split so the same partition can later support classification.

### What you should observe and learn

- No patient may appear in more than one split.
- Training data fit the point model.
- Calibration outcomes determine the uncertainty threshold.
- Test outcomes remain untouched until final evaluation.
- Reusing calibration patients for model selection would blur these roles and weaken the interpretation of the final coverage estimate.


In [ ]:
# Task 1 — Create reproducible stratified train/calibration/test splits
# ============================================================

def stratified_three_way_split(labels, train_fraction=0.50, cal_fraction=0.25, seed=SEED):
    '''Return stratified row indices for training, calibration, and testing.'''
    local_rng = np.random.default_rng(seed)
    train_parts, cal_parts, test_parts = [], [], []

    for class_id in np.unique(labels):
        class_rows = np.flatnonzero(labels == class_id)
        local_rng.shuffle(class_rows)

        n_class = len(class_rows)
        n_train_class = int(np.floor(train_fraction * n_class))
        n_cal_class = int(np.floor(cal_fraction * n_class))

        train_parts.append(class_rows[:n_train_class])
        cal_parts.append(class_rows[n_train_class:n_train_class + n_cal_class])
        test_parts.append(class_rows[n_train_class + n_cal_class:])

    return (
        np.sort(np.concatenate(train_parts)),
        np.sort(np.concatenate(cal_parts)),
        np.sort(np.concatenate(test_parts)),
    )


# WriteYourCodeHere — STUDENT VERSION:
# Call the splitting function and keep calibration separate from training/test.
# Completed answer is included below so this instructor version runs end to end.
train_idx, cal_idx, test_idx = stratified_three_way_split(
    labels=cohort["subtype"].to_numpy(),
    train_fraction=0.50,
    cal_fraction=0.25,
    seed=SEED,
)

split_sizes = pd.Series({
    "Training": len(train_idx),
    "Calibration": len(cal_idx),
    "Test": len(test_idx),
})
display(split_sizes.to_frame("Patients"))

split_balance = pd.DataFrame({
    "Training": cohort.iloc[train_idx]["subtype_name"].value_counts(normalize=True),
    "Calibration": cohort.iloc[cal_idx]["subtype_name"].value_counts(normalize=True),
    "Test": cohort.iloc[test_idx]["subtype_name"].value_counts(normalize=True),
}).reindex(SUBTYPE_NAMES)
display(split_balance.round(3))

# Automatic leakage checks.
assert len(np.intersect1d(train_idx, cal_idx)) == 0
assert len(np.intersect1d(train_idx, test_idx)) == 0
assert len(np.intersect1d(cal_idx, test_idx)) == 0
assert len(np.union1d(np.union1d(train_idx, cal_idx), test_idx)) == len(cohort)
print("Checkpoint passed: every patient has exactly one data role.")


## Task 2 — Turn Calibration Residuals into Prediction Intervals

### Task background

The fitted regression model returns one estimated phenotype value per patient. The translational team now needs an uncertainty range that describes the procedure's expected error on future patients drawn from the same cohort process.

### What you will do

1. fit the provided point model using only training patients;
2. calculate absolute residuals on calibration patients;
3. select the finite-sample conformal quantile;
4. construct intervals for untouched test patients;
5. compare point-model RMSE, empirical coverage, and mean interval width.

### What you should observe and learn

- Test coverage should be close to the target $1-\alpha$, but finite test samples need not equal it exactly.
- Lower $\alpha$ requests higher coverage and produces wider intervals.
- RMSE evaluates point accuracy, whereas coverage and width evaluate the uncertainty procedure.
- The interval is centered on the model prediction but its half-width comes from held-out calibration errors.


In [ ]:
# Part 1.1 — Small NumPy regression utilities
# ============================================================
# The point model is deliberately simple: conformal prediction wraps around
# a fitted model; it does not require a specific regression algorithm.

GENE_COLUMNS = [f"gene_{gene_id:02d}" for gene_id in range(N_GENES)]
BASE_FEATURE_COLUMNS = GENE_COLUMNS + ["age", "group_code", "site"]


def make_regression_matrix(frame):
    '''Create model features, including a few nonlinear summaries.'''
    base = frame[BASE_FEATURE_COLUMNS].to_numpy(dtype=float)
    genes = frame[GENE_COLUMNS].to_numpy(dtype=float)

    # Low-order summaries let a linear model capture part of the nonlinear signal.
    summaries = np.column_stack([
        genes[:, :8].mean(axis=1),
        genes[:, 8:16].mean(axis=1),
        genes[:, 16:24].mean(axis=1),
        genes[:, 24:32].mean(axis=1),
        genes[:, 32:40].mean(axis=1),
        genes[:, 24:32].mean(axis=1) * genes[:, 32:40].mean(axis=1),
    ])
    return np.column_stack([base, summaries])


def fit_ridge_regression(X, y, penalty=8.0):
    '''Fit standardized Ridge regression and return all preprocessing values.'''
    x_mean = X.mean(axis=0)
    x_std = X.std(axis=0)
    x_std[x_std == 0] = 1.0
    Xz = (X - x_mean) / x_std

    y_mean = y.mean()
    yc = y - y_mean
    identity = np.eye(Xz.shape[1])
    beta = np.linalg.solve(Xz.T @ Xz + penalty * identity, Xz.T @ yc)
    return {"x_mean": x_mean, "x_std": x_std, "y_mean": y_mean, "beta": beta}


def predict_ridge(model, X):
    '''Apply a fitted standardized Ridge model.'''
    Xz = (X - model["x_mean"]) / model["x_std"]
    return model["y_mean"] + Xz @ model["beta"]


def rmse(y_true, y_pred):
    '''Return root mean squared error.'''
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


X_all_reg = make_regression_matrix(cohort)
y_all_reg = cohort["phenotype"].to_numpy()

X_train_reg, y_train_reg = X_all_reg[train_idx], y_all_reg[train_idx]
X_cal_reg, y_cal_reg = X_all_reg[cal_idx], y_all_reg[cal_idx]
X_test_reg, y_test_reg = X_all_reg[test_idx], y_all_reg[test_idx]

point_model = fit_ridge_regression(X_train_reg, y_train_reg, penalty=8.0)
pred_train_reg = predict_ridge(point_model, X_train_reg)
pred_cal_reg = predict_ridge(point_model, X_cal_reg)
pred_test_reg = predict_ridge(point_model, X_test_reg)

print(f"Training RMSE:   {rmse(y_train_reg, pred_train_reg):.3f}")
print(f"Calibration RMSE:{rmse(y_cal_reg, pred_cal_reg):.3f}")
print(f"Test RMSE:       {rmse(y_test_reg, pred_test_reg):.3f}")


In [ ]:
# Task 2 — Implement split conformal regression
# ============================================================

ALPHA = 0.10


def conformal_quantile(scores, alpha):
    '''Return the finite-sample corrected split-conformal quantile.'''
    scores = np.asarray(scores, dtype=float)
    n_scores = len(scores)

    # WriteYourCodeHere — STUDENT VERSION:
    # Compute ceil((n + 1) * (1 - alpha)).
    # Completed answer:
    rank = int(np.ceil((n_scores + 1) * (1 - alpha)))
    if rank > n_scores:
        return float("inf")
    rank = max(rank, 1)

    # np.partition finds the requested order statistic without fully sorting.
    return float(np.partition(scores, rank - 1)[rank - 1])


def interval_metrics(y_true, lower, upper):
    '''Calculate marginal coverage and mean interval width.'''
    y_true = np.asarray(y_true)
    covered = (y_true >= lower) & (y_true <= upper)
    return {
        "coverage": float(covered.mean()),
        "mean_width": float(np.mean(upper - lower)),
        "covered": covered,
    }


# WriteYourCodeHere — STUDENT VERSION:
# 1) calculate absolute calibration residuals;
# 2) obtain q_hat with conformal_quantile;
# 3) add/subtract q_hat from test predictions.
# Completed answer:
calibration_scores = np.abs(y_cal_reg - pred_cal_reg)
q_hat = conformal_quantile(calibration_scores, alpha=ALPHA)
fixed_lower = pred_test_reg - q_hat
fixed_upper = pred_test_reg + q_hat

fixed_metrics = interval_metrics(y_test_reg, fixed_lower, fixed_upper)

print(f"Requested coverage: {1 - ALPHA:.1%}")
print(f"Conformal quantile: {q_hat:.3f}")
print(f"Observed coverage:  {fixed_metrics['coverage']:.1%}")
print(f"Mean interval width:{fixed_metrics['mean_width']:.3f}")

assert np.all(fixed_lower <= fixed_upper)
assert 0.0 <= fixed_metrics["coverage"] <= 1.0
print("Checkpoint passed: valid fixed-width test intervals were constructed.")


In [ ]:
# Task 2 — Visualize patient-level intervals and the coverage-width trade-off
# ============================================================

# Sort a subset by point prediction so interval failures are easy to locate.
display_order = np.argsort(pred_test_reg)[:80]
x_position = np.arange(len(display_order))
covered_subset = fixed_metrics["covered"][display_order]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for is_covered, color, label in [
    (True, COLORS["blue"], "Covered"),
    (False, COLORS["red"], "Missed"),
]:
    mask = covered_subset == is_covered
    rows = display_order[mask]
    axes[0].errorbar(
        x_position[mask],
        pred_test_reg[rows],
        yerr=np.vstack([
            pred_test_reg[rows] - fixed_lower[rows],
            fixed_upper[rows] - pred_test_reg[rows],
        ]),
        fmt="o",
        markersize=3,
        alpha=0.72,
        color=color,
        label=label,
    )
    axes[0].scatter(
        x_position[mask], y_test_reg[rows],
        marker="x", s=24, color="black", alpha=0.75,
    )

axes[0].set_title("90% split-conformal intervals for 80 test patients")
axes[0].set_xlabel("Patients sorted by predicted phenotype")
axes[0].set_ylabel("Phenotype")
axes[0].legend()

alpha_grid = np.array([0.05, 0.10, 0.15, 0.20, 0.30])
tradeoff_rows = []
for alpha_value in alpha_grid:
    q_value = conformal_quantile(calibration_scores, alpha_value)
    lower_value = pred_test_reg - q_value
    upper_value = pred_test_reg + q_value
    metrics_value = interval_metrics(y_test_reg, lower_value, upper_value)
    tradeoff_rows.append({
        "alpha": alpha_value,
        "target_coverage": 1 - alpha_value,
        "observed_coverage": metrics_value["coverage"],
        "mean_width": metrics_value["mean_width"],
    })

tradeoff_table = pd.DataFrame(tradeoff_rows)
axes[1].plot(
    tradeoff_table["mean_width"], tradeoff_table["observed_coverage"],
    "o-", color=COLORS["purple"], label="Observed",
)
for _, row in tradeoff_table.iterrows():
    axes[1].annotate(
        f"α={row['alpha']:.2f}",
        (row["mean_width"], row["observed_coverage"]),
        xytext=(5, 4), textcoords="offset points", fontsize=8,
    )
axes[1].set_title("Higher requested coverage requires wider intervals")
axes[1].set_xlabel("Mean interval width")
axes[1].set_ylabel("Observed test coverage")
axes[1].set_ylim(0.60, 1.01)

plt.tight_layout()
plt.show()

display(tradeoff_table.round(3))


### Parameter experiment — Change the requested coverage

Change the clearly marked `ALPHA_CONTROL` value below and rerun that code cell. This changes $\alpha$ without refitting the regression model. The calibration residuals remain fixed; only the conformal quantile and the resulting test intervals change.

- Smaller $\alpha$ means higher requested coverage.
- Higher requested coverage selects a larger residual quantile.
- A larger quantile produces wider intervals.

Try `0.05`, `0.10`, `0.20`, and `0.30`. The rest of the notebook continues to use the main setting `ALPHA = 0.10`, so this experiment does not silently change later Tasks.


In [ ]:
# Task 2 — Interactive coverage-versus-width experiment
# ============================================================

# WriteYourCodeHere — STUDENT CONTROL:
# Change this value and rerun the cell to update the experiment.
ALPHA_CONTROL = 0.10


def explore_regression_alpha(alpha=ALPHA_CONTROL, show_plot=True):
    '''Recalculate split-conformal intervals for one selected alpha.'''
    if not 0.01 <= alpha <= 0.40:
        raise ValueError("alpha must be between 0.01 and 0.40 for this experiment.")

    q_value = conformal_quantile(calibration_scores, alpha=float(alpha))
    lower_value = pred_test_reg - q_value
    upper_value = pred_test_reg + q_value
    metrics_value = interval_metrics(y_test_reg, lower_value, upper_value)

    summary = pd.Series({
        "alpha": float(alpha),
        "requested coverage": 1 - float(alpha),
        "observed test coverage": metrics_value["coverage"],
        "mean interval width": metrics_value["mean_width"],
        "conformal quantile": q_value,
    })

    if show_plot:
        rows = np.argsort(pred_test_reg)[:55]
        positions = np.arange(len(rows))
        covered = metrics_value["covered"][rows]

        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
        axes[0].errorbar(
            positions,
            pred_test_reg[rows],
            yerr=q_value,
            fmt="o",
            markersize=3,
            alpha=0.65,
            color=COLORS["blue"],
            label="Prediction interval",
        )
        axes[0].scatter(
            positions[covered], y_test_reg[rows][covered],
            marker="x", s=25, color="black", label="Covered outcome",
        )
        axes[0].scatter(
            positions[~covered], y_test_reg[rows][~covered],
            marker="x", s=38, color=COLORS["red"], label="Missed outcome",
        )
        axes[0].set_title(
            f"Requested {1-alpha:.0%} | observed {metrics_value['coverage']:.1%}"
        )
        axes[0].set_xlabel("55 patients sorted by prediction")
        axes[0].set_ylabel("Phenotype")
        axes[0].legend(fontsize=8)

        axes[1].hist(
            calibration_scores, bins=32,
            color=COLORS["purple"], alpha=0.75,
        )
        axes[1].axvline(
            q_value, color=COLORS["red"], linestyle="--",
            label=f"q = {q_value:.2f}",
        )
        axes[1].set_title("Calibration residuals determine interval width")
        axes[1].set_xlabel("Absolute calibration residual")
        axes[1].set_ylabel("Patients")
        axes[1].legend()

        plt.tight_layout()
        plt.show()

    display(summary.to_frame("Value").round(3))
    return summary


# Always create a static result so the notebook remains informative everywhere.
alpha_snapshot = explore_regression_alpha(ALPHA_CONTROL, show_plot=True)

print("Control ready: change ALPHA_CONTROL above and rerun this cell.")
print("Suggested values: 0.05, 0.10, 0.20, 0.30")


# Part 2 — Adaptive Intervals and Subgroup Reliability

## Concept — Heteroscedasticity, Local Scale, and Marginal Blind Spots

### Why one global width can be inefficient

The interval from Part 1 uses the same half-width $q$ for every patient. This is sensible when residual variability is roughly constant. Biomedical data often violate that condition: measurement error may change with assay intensity, disease severity, clinical site, ancestry composition, or the density of similar training examples.

This feature-dependent variability is called **heteroscedasticity**:

$$
\operatorname{Var}(Y\mid X=x)
$$

changes with $x$. A global interval may then be unnecessarily wide in low-noise regions and too narrow in high-noise regions, even while its average coverage remains near the nominal target.

### Normalize before calibrating

Suppose a scale model estimates the typical residual magnitude $\widehat\sigma(x)>0$. We can define

$$
R_i=\frac{|Y_i-\widehat\mu(X_i)|}{\widehat\sigma(X_i)}.
$$

After calibrating a quantile $q$ of these normalized scores, the patient-specific interval is

$$
C(x)=\left[\widehat\mu(x)-q\widehat\sigma(x),\;
\widehat\mu(x)+q\widehat\sigma(x)\right].
$$

The same calibrated multiplier is used for everyone, but the predicted scale redistributes width. Difficult cases receive wider intervals; easier cases can receive narrower ones.

### Why the scale model does not need to be perfect

The point and scale models are fixed using training data before calibration begins. Conditional on those fitted models, calibration and test scores can still be compared exchangeably. A poor scale model mainly damages **efficiency**: widths may vary in unhelpful ways. Calibration still corrects the overall score threshold when its assumptions hold.

In this notebook the scale model learns from training residuals only. Calibration outcomes do not fit the scale model.

### Overall coverage can hide unequal reliability

Marginal coverage is an average. For example, 95% coverage in a large subgroup and 75% coverage in a smaller subgroup can combine into an apparently acceptable overall result. We therefore report, for each population group:

- number of test patients;
- empirical coverage;
- mean interval width.

This is a **subgroup audit**, not a proof of conditional coverage. Small groups produce noisier empirical estimates, and inspecting many groups after seeing results can encourage selective reporting.

---

## Task 3 — Audit Whom the Intervals Protect

### Task background

In the simulated cohort, phenotype noise changes with hypoxia activity and population group. The translational team wants to know whether the fixed-width interval protects all groups similarly and whether a learned residual scale can allocate interval width more appropriately.

### What you will do

1. fit the provided residual-scale model using training residuals;
2. normalize calibration residuals by predicted scale;
3. construct adaptive test intervals;
4. compare fixed and adaptive intervals overall and within each population group.

### What you should observe and learn

- Overall 90% coverage can coexist with subgroup undercoverage.
- Adaptive intervals become wider for patients with larger predicted error scales.
- Improved subgroup balance may require different interval widths across groups.
- A subgroup table is necessary evidence, but it does not convert a marginal guarantee into a universal conditional guarantee.


In [ ]:
# Part 2.1 — Fit a residual-scale model using training data only
# ============================================================
# We model log absolute residuals to keep predicted scales positive.
# The calibration outcomes are not used to fit either model.

training_abs_residual = np.abs(y_train_reg - pred_train_reg)
log_scale_target = np.log(training_abs_residual + 0.50)

scale_model = fit_ridge_regression(
    X_train_reg,
    log_scale_target,
    penalty=20.0,
)


def predict_residual_scale(model, X):
    '''Return positive residual-scale predictions with defensive clipping.'''
    predicted_log_scale = predict_ridge(model, X)
    return np.clip(np.exp(predicted_log_scale), 0.75, 20.0)


scale_cal = predict_residual_scale(scale_model, X_cal_reg)
scale_test = predict_residual_scale(scale_model, X_test_reg)

print("Predicted calibration-scale range:", tuple(np.round([scale_cal.min(), scale_cal.max()], 3)))
print("Predicted test-scale range:       ", tuple(np.round([scale_test.min(), scale_test.max()], 3)))


In [ ]:
# Task 3 — Construct adaptive intervals and audit population groups
# ============================================================

# WriteYourCodeHere — STUDENT VERSION:
# Normalize calibration errors by predicted scale, calibrate one quantile,
# and multiply that quantile by each test patient's predicted scale.
# Completed answer:
normalized_calibration_scores = calibration_scores / scale_cal
q_normalized = conformal_quantile(normalized_calibration_scores, alpha=ALPHA)
adaptive_half_width = q_normalized * scale_test
adaptive_lower = pred_test_reg - adaptive_half_width
adaptive_upper = pred_test_reg + adaptive_half_width

adaptive_metrics = interval_metrics(y_test_reg, adaptive_lower, adaptive_upper)


def subgroup_interval_audit(frame, y_true, methods):
    '''Compare interval coverage and width for every population group.'''
    rows = []
    groups = frame["population_group"].to_numpy()

    for method_name, (lower, upper) in methods.items():
        for group_name in GROUP_NAMES:
            mask = groups == group_name
            metrics = interval_metrics(y_true[mask], lower[mask], upper[mask])
            rows.append({
                "Method": method_name,
                "Group": group_name,
                "Patients": int(mask.sum()),
                "Coverage": metrics["coverage"],
                "Mean width": metrics["mean_width"],
            })
    return pd.DataFrame(rows)


test_frame = cohort.iloc[test_idx].reset_index(drop=True)
subgroup_audit = subgroup_interval_audit(
    test_frame,
    y_test_reg,
    {
        "Fixed": (fixed_lower, fixed_upper),
        "Adaptive": (adaptive_lower, adaptive_upper),
    },
)

overall_comparison = pd.DataFrame([
    {
        "Method": "Fixed",
        "Coverage": fixed_metrics["coverage"],
        "Mean width": fixed_metrics["mean_width"],
    },
    {
        "Method": "Adaptive",
        "Coverage": adaptive_metrics["coverage"],
        "Mean width": adaptive_metrics["mean_width"],
    },
]).set_index("Method")

display(overall_comparison.round(3))
display(subgroup_audit.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
coverage_pivot = subgroup_audit.pivot(index="Group", columns="Method", values="Coverage")
width_pivot = subgroup_audit.pivot(index="Group", columns="Method", values="Mean width")

coverage_pivot.plot.bar(ax=axes[0], color=[COLORS["green"], COLORS["blue"]])
axes[0].axhline(1 - ALPHA, color="black", linestyle="--", label="90% target")
axes[0].set_title("Coverage must be audited by group")
axes[0].set_ylabel("Coverage")
axes[0].set_ylim(0.65, 1.01)
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend()

width_pivot.plot.bar(ax=axes[1], color=[COLORS["green"], COLORS["blue"]])
axes[1].set_title("Adaptive intervals redistribute width")
axes[1].set_ylabel("Mean interval width")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

assert np.std(adaptive_upper - adaptive_lower) > 0
print("Checkpoint passed: adaptive intervals vary across patients.")


# Part 3 — Classification Prediction Sets

## Concept — From Forced Top-1 Decisions to Calibrated Sets

### A probability vector is not yet a coverage guarantee

A multiclass classifier returns probabilities

$$
\widehat\pi_1(x),\ldots,\widehat\pi_K(x),
\qquad \sum_{k=1}^{K}\widehat\pi_k(x)=1.
$$

The top-1 rule selects the largest probability, even when the leading classes are nearly tied. Softmax values may also be miscalibrated, so a reported probability of 0.9 does not by itself guarantee 90% correctness.

Conformal classification changes the output from one forced label to a **prediction set** $C(x)\subseteq\{1,\ldots,K\}$. The desired marginal property is

$$
\Pr\{Y_{\mathrm{new}}\in C(X_{\mathrm{new}})\}\geq1-\alpha.
$$

### Cumulative-probability scores

This notebook uses a deterministic adaptive prediction-set score. For each patient, sort class probabilities from largest to smallest. If candidate label $y$ appears at rank $r$, define

$$
S(x,y)=\sum_{j=1}^{r}\widehat\pi_{(j)}(x).
$$

A highly ranked label receives a smaller cumulative score. A low-ranked label can be included only when the calibrated threshold is large enough. Calibration extracts the score of the true label for each calibration patient and selects a high conformal quantile.

At prediction time, every candidate label satisfying

$$
S(x,y)\leq q
$$

is retained. We also keep the top-1 label so the set is never empty; enlarging a set cannot reduce coverage.

### Coverage versus set size

For classification, efficiency is measured by quantities such as:

- mean prediction-set size;
- fraction of singleton sets;
- frequency of very large sets.

A singleton is decisive, whereas a multi-label set explicitly communicates ambiguity. A set containing every subtype is safe but rarely useful.

### Marginal and class-conditional calibration

A single global threshold targets marginal coverage across all classes. A common class may dominate this average, allowing a rare or difficult class to be undercovered.

Class-conditional, or Mondrian-style, calibration instead estimates one threshold $q_k$ from calibration patients whose true label is class $k$. Candidate label $k$ is then compared with $q_k$. This aims to balance coverage by true class, but it has a statistical cost: each threshold uses fewer calibration examples. Rare classes therefore have coarser and more variable thresholds.

---

## Task 4 — Replace One Cancer Label with a Set of Plausible Labels

### Task background

The research group predicts one of five simulated cancer subtypes from gene-expression measurements. In a clinical workflow, a calibrated set of plausible labels resembles a differential diagnosis more closely than a forced top-1 output:

~~~text
{LUAD-like}                     confident singleton
{LUAD-like, KIRC-like}          ambiguous differential
{BRCA-like, COAD-like, ...}     broad uncertainty
~~~

### What you will do

1. use the provided softmax classifier to obtain class probabilities;
2. calculate cumulative-probability scores for every candidate label;
3. calibrate a global prediction-set threshold;
4. calibrate separate class-conditional thresholds;
5. compare accuracy, coverage, set size, singleton rate, and per-class coverage.

### What you should observe and learn

- Accuracy and set coverage answer different questions.
- Larger sets are less decisive but can cover the true label more often.
- Marginal coverage can conceal a difficult or rare undercovered class.
- Class-conditional calibration can redistribute protection, but its per-class estimates are noisier.


In [ ]:
# Part 3.1 — Fit a compact NumPy softmax classifier
# ============================================================
# Model training is provided because the learning target is conformal
# calibration, not optimization of multinomial logistic regression.

def softmax(logits):
    '''Convert logits into row-wise probabilities stably.'''
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def fit_softmax_classifier(X, y, n_classes, learning_rate=0.08, l2=0.04, n_steps=1800):
    '''Fit standardized multinomial logistic regression by gradient descent.'''
    x_mean = X.mean(axis=0)
    x_std = X.std(axis=0)
    x_std[x_std == 0] = 1.0
    Xz = (X - x_mean) / x_std
    X_design = np.column_stack([np.ones(len(Xz)), Xz])

    weights = np.zeros((X_design.shape[1], n_classes))
    y_one_hot = np.eye(n_classes)[y]

    for _ in range(n_steps):
        probabilities = softmax(X_design @ weights)
        gradient = X_design.T @ (probabilities - y_one_hot) / len(y)
        gradient[1:] += l2 * weights[1:]
        weights -= learning_rate * gradient

    return {"x_mean": x_mean, "x_std": x_std, "weights": weights}


def predict_softmax_proba(model, X):
    '''Return class probabilities from a fitted softmax model.'''
    Xz = (X - model["x_mean"]) / model["x_std"]
    X_design = np.column_stack([np.ones(len(Xz)), Xz])
    return softmax(X_design @ model["weights"])


X_all_cls = cohort[GENE_COLUMNS].to_numpy(dtype=float)
y_all_cls = cohort["subtype"].to_numpy(dtype=int)

X_train_cls, y_train_cls = X_all_cls[train_idx], y_all_cls[train_idx]
X_cal_cls, y_cal_cls = X_all_cls[cal_idx], y_all_cls[cal_idx]
X_test_cls, y_test_cls = X_all_cls[test_idx], y_all_cls[test_idx]

classifier = fit_softmax_classifier(
    X_train_cls,
    y_train_cls,
    n_classes=len(SUBTYPE_NAMES),
)
proba_cal = predict_softmax_proba(classifier, X_cal_cls)
proba_test = predict_softmax_proba(classifier, X_test_cls)

accuracy_cal = np.mean(np.argmax(proba_cal, axis=1) == y_cal_cls)
accuracy_test = np.mean(np.argmax(proba_test, axis=1) == y_test_cls)

print(f"Calibration top-1 accuracy: {accuracy_cal:.1%}")
print(f"Test top-1 accuracy:        {accuracy_test:.1%}")
assert np.allclose(proba_test.sum(axis=1), 1.0)


In [ ]:
# Task 4 — Implement cumulative-probability conformal prediction sets
# ============================================================

def cumulative_score_matrix(probabilities):
    '''Return the cumulative sorted-probability score for every candidate label.'''
    probabilities = np.asarray(probabilities, dtype=float)
    order = np.argsort(-probabilities, axis=1)
    sorted_probabilities = np.take_along_axis(probabilities, order, axis=1)
    cumulative_sorted = np.cumsum(sorted_probabilities, axis=1)

    # WriteYourCodeHere — STUDENT VERSION:
    # Map cumulative scores from sorted positions back to original labels.
    # Completed answer:
    score_matrix = np.empty_like(cumulative_sorted)
    np.put_along_axis(score_matrix, order, cumulative_sorted, axis=1)
    return score_matrix


def prediction_sets_from_thresholds(probabilities, thresholds):
    '''Include every candidate label whose cumulative score is below its threshold.'''
    score_matrix = cumulative_score_matrix(probabilities)
    thresholds = np.asarray(thresholds, dtype=float)
    if thresholds.ndim == 0:
        thresholds = np.repeat(thresholds, probabilities.shape[1])

    # WriteYourCodeHere — STUDENT VERSION:
    # Compare each label score with its marginal or class-specific threshold.
    # Completed answer:
    prediction_sets = score_matrix <= thresholds[None, :]

    # Always retaining top-1 can only enlarge a set, so it cannot reduce coverage.
    top_label = np.argmax(probabilities, axis=1)
    prediction_sets[np.arange(len(probabilities)), top_label] = True
    return prediction_sets


def classification_set_metrics(prediction_sets, y_true):
    '''Summarize coverage, set size, and singleton frequency.'''
    row_ids = np.arange(len(y_true))
    covered = prediction_sets[row_ids, y_true]
    sizes = prediction_sets.sum(axis=1)
    return {
        "coverage": float(covered.mean()),
        "mean_size": float(sizes.mean()),
        "singleton_rate": float(np.mean(sizes == 1)),
        "covered": covered,
        "sizes": sizes,
    }


cal_score_matrix = cumulative_score_matrix(proba_cal)

# WriteYourCodeHere — STUDENT VERSION:
# Extract the true-label calibration score and calibrate one marginal threshold.
# Completed answer:
true_calibration_scores_cls = cal_score_matrix[np.arange(len(y_cal_cls)), y_cal_cls]
q_marginal_cls = conformal_quantile(true_calibration_scores_cls, alpha=ALPHA)
marginal_sets = prediction_sets_from_thresholds(proba_test, q_marginal_cls)
marginal_set_metrics = classification_set_metrics(marginal_sets, y_test_cls)

# Class-conditional thresholds use calibration examples from each true class.
# Each candidate label is compared with the threshold for that label.
q_by_class = np.array([
    conformal_quantile(
        true_calibration_scores_cls[y_cal_cls == class_id],
        alpha=ALPHA,
    )
    for class_id in range(len(SUBTYPE_NAMES))
])
class_conditional_sets = prediction_sets_from_thresholds(proba_test, q_by_class)
class_conditional_metrics = classification_set_metrics(
    class_conditional_sets, y_test_cls
)

set_comparison = pd.DataFrame([
    {
        "Method": "Marginal",
        "Coverage": marginal_set_metrics["coverage"],
        "Mean set size": marginal_set_metrics["mean_size"],
        "Singleton rate": marginal_set_metrics["singleton_rate"],
    },
    {
        "Method": "Class-conditional",
        "Coverage": class_conditional_metrics["coverage"],
        "Mean set size": class_conditional_metrics["mean_size"],
        "Singleton rate": class_conditional_metrics["singleton_rate"],
    },
]).set_index("Method")
display(set_comparison.round(3))


In [ ]:
# Task 4 — Audit per-class coverage and inspect patient-level sets
# ============================================================

class_rows = []
for class_id, class_name in enumerate(SUBTYPE_NAMES):
    mask = y_test_cls == class_id
    marginal_class_metrics = classification_set_metrics(
        marginal_sets[mask], y_test_cls[mask]
    )
    conditional_class_metrics = classification_set_metrics(
        class_conditional_sets[mask], y_test_cls[mask]
    )
    class_rows.extend([
        {
            "Class": class_name,
            "Method": "Marginal",
            "Patients": int(mask.sum()),
            "Coverage": marginal_class_metrics["coverage"],
            "Mean set size": marginal_class_metrics["mean_size"],
        },
        {
            "Class": class_name,
            "Method": "Class-conditional",
            "Patients": int(mask.sum()),
            "Coverage": conditional_class_metrics["coverage"],
            "Mean set size": conditional_class_metrics["mean_size"],
        },
    ])

class_audit = pd.DataFrame(class_rows)
display(class_audit.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
class_audit.pivot(index="Class", columns="Method", values="Coverage").plot.bar(
    ax=axes[0], color=[COLORS["green"], COLORS["blue"]]
)
axes[0].axhline(1 - ALPHA, color="black", linestyle="--", label="90% target")
axes[0].set_title("Marginal coverage can hide class differences")
axes[0].set_ylabel("Coverage")
axes[0].set_ylim(0.60, 1.01)
axes[0].tick_params(axis="x", rotation=35)
axes[0].legend()

size_counts = pd.Series(marginal_set_metrics["sizes"]).value_counts().sort_index()
axes[1].bar(size_counts.index.astype(str), size_counts.values, color=COLORS["purple"])
axes[1].set_title("Marginal prediction-set sizes")
axes[1].set_xlabel("Number of included labels")
axes[1].set_ylabel("Test patients")

plt.tight_layout()
plt.show()

# Display several patients from smallest to largest marginal sets.
example_rows = np.argsort(marginal_set_metrics["sizes"])[
    np.linspace(0, len(y_test_cls) - 1, 8, dtype=int)
]
examples = []
for row in example_rows:
    labels_in_set = SUBTYPE_NAMES[marginal_sets[row]].tolist()
    examples.append({
        "True subtype": SUBTYPE_NAMES[y_test_cls[row]],
        "Top-1 subtype": SUBTYPE_NAMES[np.argmax(proba_test[row])],
        "Top-1 probability": np.max(proba_test[row]),
        "Marginal prediction set": ", ".join(labels_in_set),
        "Set size": int(marginal_sets[row].sum()),
    })
display(pd.DataFrame(examples).round(3))

assert marginal_sets.shape == proba_test.shape
assert np.all(marginal_sets.sum(axis=1) >= 1)
print("Checkpoint passed: every patient receives a nonempty prediction set.")


# Part 4 — Distribution-Shift Stress Test

## Concept — Why Exchangeability Can Fail After Deployment

### The original guarantee has a target population

Calibration scores represent future errors only when calibration and test examples are exchangeable. Deployment can break this relationship. “Distribution shift” is not one single phenomenon, so it is useful to distinguish several mechanisms.

### Common forms of biomedical shift

| Shift type | What changes? | Biomedical example |
| --- | --- | --- |
| Covariate shift | $P(X)$ changes | A new hospital serves an older population |
| Label shift | $P(Y)$ changes | Cancer-subtype prevalence differs by referral center |
| Conditional/concept shift | $P(Y\mid X)$ changes | Treatment practice changes the outcome for similar measurements |
| Measurement shift | Recorded $X$ changes | A new scanner, assay, staining protocol, or sequencing batch |

These mechanisms can occur together. The simulated new site changes subtype prevalence, group composition, gene-expression measurements, and phenotype noise.

### Why fixed calibration may stop representing future errors

The original threshold $q$ is an order statistic of source-cohort scores. Under shift, future scores may move toward larger values or change shape. The old threshold then cuts the new score distribution at the wrong location, and empirical coverage can fall below $1-\alpha$.

Distribution-free therefore means that no parametric residual distribution is required under the conformal assumptions. It does **not** mean that one calibration set protects every future hospital or time period.

### A valid stress test freezes the deployed system

To evaluate robustness honestly:

1. fit the model on the source training set;
2. fix the conformal threshold using the source calibration set;
3. do not tune either object using shifted outcomes;
4. vary a prespecified shift mechanism;
5. report coverage and efficiency together.

If the model or threshold is repeatedly adjusted after viewing shifted test labels, the exercise is no longer an external stress test.

### What can be done after detecting shift

Possible responses include collecting recent labeled calibration data, recalibrating on the target site, weighting calibration examples when the shift mechanism is estimable, or using sequential monitoring. These are new procedures with their own assumptions. Simply widening every interval after seeing a failure is not a principled guarantee.

---

## Task 5 — Send the Model to a New Clinical Site

### Task background

The model and conformal thresholds were developed using the source consortium. A new hospital has different subtype and population-group proportions, a site-dependent gene-expression batch effect, and noisier phenotype measurements.

### What you will do

Keep the trained models and source calibration thresholds fixed. Change `SHIFT_STRENGTH_CONTROL` and rerun the Task 5 code cell to generate future cohorts and measure regression coverage, classification-set coverage, interval width, and set size.

### What you should observe and learn

- Coverage deterioration is not automatically a coding error; it may reveal a violated assumption.
- Adaptive intervals may widen under shift without fully restoring nominal coverage.
- Prediction sets can become less reliable even when their average size changes only modestly.
- Deployment requires shift monitoring and, when justified, fresh calibration data.


In [ ]:
# Task 5 — Interactive new-site stress test
# ============================================================

# WriteYourCodeHere — STUDENT CONTROL:
# Change this value and rerun the cell to update the experiment.
SHIFT_STRENGTH_CONTROL = 1.0


def explore_distribution_shift(shift_strength=SHIFT_STRENGTH_CONTROL, show_plot=True):
    '''Evaluate frozen source models and thresholds on one shifted cohort.'''
    if not 0.0 <= shift_strength <= 1.5:
        raise ValueError("shift_strength must be between 0.0 and 1.5.")

    shifted = generate_biomedical_cohort(
        n_patients=800,
        seed=2026,
        shift_strength=float(shift_strength),
    )

    # Regression uses the original source point model, scale model, and thresholds.
    X_shift_reg = make_regression_matrix(shifted)
    y_shift_reg = shifted["phenotype"].to_numpy()
    pred_shift_reg = predict_ridge(point_model, X_shift_reg)
    scale_shift = predict_residual_scale(scale_model, X_shift_reg)

    fixed_shift_metrics = interval_metrics(
        y_shift_reg,
        pred_shift_reg - q_hat,
        pred_shift_reg + q_hat,
    )
    adaptive_shift_metrics = interval_metrics(
        y_shift_reg,
        pred_shift_reg - q_normalized * scale_shift,
        pred_shift_reg + q_normalized * scale_shift,
    )

    # Classification also keeps its source classifier and threshold frozen.
    X_shift_cls = shifted[GENE_COLUMNS].to_numpy(dtype=float)
    y_shift_cls = shifted["subtype"].to_numpy(dtype=int)
    proba_shift = predict_softmax_proba(classifier, X_shift_cls)
    shifted_sets = prediction_sets_from_thresholds(proba_shift, q_marginal_cls)
    shifted_set_metrics = classification_set_metrics(shifted_sets, y_shift_cls)

    summary = pd.DataFrame([
        {
            "Output": "Fixed regression interval",
            "Source coverage": fixed_metrics["coverage"],
            "Shifted coverage": fixed_shift_metrics["coverage"],
            "Source efficiency": fixed_metrics["mean_width"],
            "Shifted efficiency": fixed_shift_metrics["mean_width"],
        },
        {
            "Output": "Adaptive regression interval",
            "Source coverage": adaptive_metrics["coverage"],
            "Shifted coverage": adaptive_shift_metrics["coverage"],
            "Source efficiency": adaptive_metrics["mean_width"],
            "Shifted efficiency": adaptive_shift_metrics["mean_width"],
        },
        {
            "Output": "Marginal classification set",
            "Source coverage": marginal_set_metrics["coverage"],
            "Shifted coverage": shifted_set_metrics["coverage"],
            "Source efficiency": marginal_set_metrics["mean_size"],
            "Shifted efficiency": shifted_set_metrics["mean_size"],
        },
    ]).set_index("Output")

    if show_plot:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
        x = np.arange(3)
        width = 0.34
        source_coverage = summary["Source coverage"].to_numpy()
        shifted_coverage = summary["Shifted coverage"].to_numpy()

        axes[0].bar(
            x - width / 2, source_coverage, width,
            label="Source", color=COLORS["blue"],
        )
        axes[0].bar(
            x + width / 2, shifted_coverage, width,
            label="Shifted", color=COLORS["red"],
        )
        axes[0].axhline(1 - ALPHA, color="black", linestyle="--", label="90% target")
        axes[0].set_xticks(x, ["Fixed\nregression", "Adaptive\nregression", "Classification\nset"])
        axes[0].set_ylim(0.50, 1.02)
        axes[0].set_ylabel("Coverage")
        axes[0].set_title(f"Coverage at shift strength = {shift_strength:.2f}")
        axes[0].legend(fontsize=8)

        source_group = cohort.iloc[test_idx]["population_group"].value_counts(normalize=True).reindex(GROUP_NAMES)
        shifted_group = shifted["population_group"].value_counts(normalize=True).reindex(GROUP_NAMES)
        composition = pd.DataFrame({"Source": source_group, "Shifted": shifted_group})
        composition.plot.bar(
            ax=axes[1], color=[COLORS["blue"], COLORS["orange"]]
        )
        axes[1].set_title("Population composition also changes")
        axes[1].set_ylabel("Patient proportion")
        axes[1].set_xlabel("")
        axes[1].tick_params(axis="x", rotation=0)

        plt.tight_layout()
        plt.show()

    display(summary.round(3))
    return summary


# Always create a static result so the notebook remains informative everywhere.
selected_shift_summary = explore_distribution_shift(
    SHIFT_STRENGTH_CONTROL,
    show_plot=True,
)

print("Control ready: change SHIFT_STRENGTH_CONTROL above and rerun this cell.")
print("Suggested values: 0.0, 0.5, 1.0, 1.5")


In [ ]:
# Task 5 — Trace coverage as shift becomes stronger
# ============================================================
# Reusing the same seed keeps latent random draws aligned across shift levels,
# so the curves primarily reflect the controlled shift mechanism.

shift_grid = np.linspace(0.0, 1.5, 7)
shift_rows = []

for strength in shift_grid:
    future = generate_biomedical_cohort(
        n_patients=900,
        seed=707,
        shift_strength=float(strength),
    )

    X_future_reg = make_regression_matrix(future)
    y_future_reg = future["phenotype"].to_numpy()
    pred_future_reg = predict_ridge(point_model, X_future_reg)
    scale_future = predict_residual_scale(scale_model, X_future_reg)

    fixed_future = interval_metrics(
        y_future_reg,
        pred_future_reg - q_hat,
        pred_future_reg + q_hat,
    )
    adaptive_future = interval_metrics(
        y_future_reg,
        pred_future_reg - q_normalized * scale_future,
        pred_future_reg + q_normalized * scale_future,
    )

    X_future_cls = future[GENE_COLUMNS].to_numpy(dtype=float)
    y_future_cls = future["subtype"].to_numpy(dtype=int)
    proba_future = predict_softmax_proba(classifier, X_future_cls)
    sets_future = prediction_sets_from_thresholds(proba_future, q_marginal_cls)
    class_future = classification_set_metrics(sets_future, y_future_cls)

    shift_rows.append({
        "Shift strength": strength,
        "Fixed regression coverage": fixed_future["coverage"],
        "Adaptive regression coverage": adaptive_future["coverage"],
        "Classification-set coverage": class_future["coverage"],
        "Adaptive mean width": adaptive_future["mean_width"],
        "Mean classification-set size": class_future["mean_size"],
    })

shift_table = pd.DataFrame(shift_rows)
display(shift_table.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for column, color in [
    ("Fixed regression coverage", COLORS["blue"]),
    ("Adaptive regression coverage", COLORS["green"]),
    ("Classification-set coverage", COLORS["purple"]),
]:
    axes[0].plot(
        shift_table["Shift strength"], shift_table[column],
        "o-", label=column.replace(" coverage", ""), color=color,
    )
axes[0].axhline(1 - ALPHA, color="black", linestyle="--", label="90% target")
axes[0].set_title("Coverage is not guaranteed under arbitrary shift")
axes[0].set_xlabel("Shift strength")
axes[0].set_ylabel("Coverage")
axes[0].set_ylim(0.55, 1.01)
axes[0].legend(fontsize=8)

axes[1].plot(
    shift_table["Shift strength"],
    shift_table["Adaptive mean width"],
    "o-", color=COLORS["orange"], label="Adaptive interval width",
)
second_axis = axes[1].twinx()
second_axis.plot(
    shift_table["Shift strength"],
    shift_table["Mean classification-set size"],
    "s--", color=COLORS["red"], label="Classification set size",
)
axes[1].set_title("Efficiency also changes after deployment")
axes[1].set_xlabel("Shift strength")
axes[1].set_ylabel("Mean interval width", color=COLORS["orange"])
second_axis.set_ylabel("Mean set size", color=COLORS["red"])

plt.tight_layout()
plt.show()

print("Checkpoint passed: the stress test keeps models and thresholds fixed.")


# Part 5 — Biomedical Interpretation and Responsible Use

## Concept — Translating Statistical Coverage into Biomedical Decisions

### A procedural guarantee is not an individual posterior probability

A 90% conformal procedure is designed so that, over repeated exchangeable future patients, at least about 90% of prediction intervals or sets contain the observed outcome. After an interval has been produced for one fixed patient, the frequentist guarantee should not be restated as “this patient's unknown outcome has a 90% probability of lying here” unless an additional probabilistic model justifies that interpretation.

### Prediction intervals, confidence intervals, and causal intervals differ

- A **prediction interval** targets a new observable outcome.
- A **confidence interval** typically targets an unknown parameter or conditional mean.
- A **causal interval** requires an intervention, estimand, and identification assumptions.

Conformal calibration of predictive residuals does not turn an associational model into a causal model.

### Statistical validity does not define a clinical action

A prediction set may support differential diagnosis, rule-out review, or escalation to a human expert, but the action depends on asymmetric clinical consequences. Missing a dangerous subtype and including an extra benign subtype do not necessarily have equal costs. The value of $\alpha$, subgroup policy, and acceptable set size must therefore be connected to a prespecified use case rather than selected only for visual convenience.

### Trustworthy reporting requires several views

No single number is sufficient. A biomedical report should combine:

- point-model discrimination or error;
- marginal coverage;
- interval width or prediction-set size;
- subgroup and class-specific audits;
- external-site or temporal evaluation;
- a clear description of the target population and calibration date.

The following table summarizes how to communicate the outputs from this notebook.

## What the Outputs Mean

| Output | Appropriate interpretation | Inappropriate interpretation |
| --- | --- | --- |
| Regression interval | A procedure designed to cover future outcomes at a target long-run rate under exchangeability | A confidence interval for the fitted mean, or a causal effect interval |
| Classification set | Labels retained by a calibrated set-valued procedure | A list of diagnoses proven correct for this individual |
| Marginal coverage | Average coverage over future examples from the same process | Equal coverage for every subgroup or patient |
| Wider interval / larger set | The procedure is less decisive for that case | The model has discovered a specific biological mechanism |

## Connection to Medical Imaging and Segmentation

The same logic can wrap around image classifiers: calibration scores transform softmax outputs into label sets. Segmentation is more complicated because the output is a structured mask rather than one scalar or one class. A segmentation extension must define what is being covered or controlled—for example pixel labels, multilabel masks, object counts, or an expected segmentation risk. Those are different guarantees and are outside the core implementation here.

## Deployment Checklist

1. Define the target population and the clinical action linked to the output.
2. Keep training, calibration, and final test data separate.
3. Report point-model performance together with coverage and efficiency.
4. Audit clinically relevant subgroups and sample sizes.
5. Test external sites, assay batches, and temporal cohorts.
6. Monitor feature, score, interval-width, and set-size distributions after deployment.
7. Collect recent labeled data and recalibrate when the deployment process changes.
8. Treat uncertainty output as decision support, not automatic clinical authority.

## Main Findings

~~~text
Residuals calibrate regression intervals.
Class-probability rankings calibrate prediction sets.
Coverage and efficiency must be reported together.
Marginal success can hide subgroup or class failures.
Distribution shift can invalidate the original guarantee.
~~~

## Reflection Questions

1. Why must the calibration patients remain separate from model training?
2. Why is the corrected conformal quantile slightly more conservative than an ordinary empirical quantile?
3. How can overall 90% coverage coexist with undercoverage in Group C?
4. Why can a prediction set be more clinically honest than a forced top-1 label?
5. What statistical price is paid when calibration is performed separately for each class?
6. Which changes in the new-site experiment violate exchangeability?
7. What new data would you request before recalibrating the deployed model?

## Scope and Limitations

This compact experiment does not establish conditional coverage for every patient, causal validity, prospective clinical utility, robustness to every distribution shift, survival-analysis coverage, time-series guarantees, federated calibration, or mask-level segmentation risk control. The cohort is simulated and should not be interpreted biologically.

> A conformal guarantee is useful only when its assumptions, target population, and efficiency are reported as clearly as its nominal coverage.
